In [1]:
import pandas as pd

In [2]:
# read indexes of data samples to use for evalution
index_path = '/data/hyeryung/mucoco/new_module/data/sentiment/dev_set_below_positive_threshold_index.jsonl'
with open(index_path, 'r') as f:
    indexes = [int(i) for i in f.read().split()]

In [5]:
# bolt
bolt_path = ['/data/hyeryung/BOLT/sentiment/sentiment/pos.len12.jsonl',
             '/data/hyeryung/BOLT/sentiment/sentiment/pos.len20.jsonl', 
             '/data/hyeryung/BOLT/sentiment/sentiment/pos.len50.jsonl']

# concat 12,20,50 results
bolt_s = []
for path in bolt_path:
    bolt_s.append(pd.read_json(path, lines=True))
bolt = pd.concat(bolt_s,axis=0,ignore_index=True)

# unravel
bolt = bolt.explode('generations').reset_index(drop=True)
bolt['generations'] = bolt['generations'].apply(lambda x: [x])

# save
# bolt.loc[indexes,].to_json('/data/hyeryung/mucoco/outputs/sentiment/bolt/clsf/pos.dev_set_below_positive_threshold.jsonl', 
                        #    orient='records', lines=True)

In [7]:
# original
# save unraveled original data
original = pd.read_json('/data/hyeryung/mucoco/new_module/data/sentiment/dev_set_below_positive_threshold_778.jsonl',lines=True)
original = original.explode('generations').reset_index(drop=True)
original['generations'] = original['generations'].apply(lambda x: [x])

# original.to_json('/data/hyeryung/mucoco/new_module/data/sentiment/dev_set_below_positive_threshold_778_unraveled.jsonl',
                    # orient='records', lines=True)

In [ ]:
# mixmatch

# get edited only indexes for mixmatch

# get mixmatch - original index mapping
# mixmatch original data (data used as original data for mixmatch)
mixmatch_original = pd.read_json('/data/hyeryung/mixmatch/data/sentiment/merged/dev_set_neg2pos.jsonl', lines=True)
mixmatch_original['prompt'] = mixmatch_original['prompt'].apply(lambda x: x['text'])
mixmatch_original['generations'] = mixmatch_original['generations'].apply(lambda x: x[0]['text'])

# original data (with all samples)
original_all = pd.read_json('/data/hyeryung/mucoco/new_module/data/sentiment/dev_set.jsonl', lines=True)
original_all = original_all.explode('generations',ignore_index=True).reset_index()
original_all['prompt'] = original_all['prompt'].apply(lambda x: x['text'])
original_all['generations'] = original_all['generations'].apply(lambda x: x['text'])

# index mapping
index_mapping = mixmatch_original.merge(original_all,on=['prompt','generations'],how='left')
index_mapping = index_mapping.rename(columns={'index': 'original_index'})
index_mapping = index_mapping.reset_index().rename(columns={'index': 'mixmatch_index'})
index_mapping = index_mapping[['original_index', 'mixmatch_index']]

# get edited indexes for mixmatch
indexes_mixmatch = index_mapping.set_index('original_index').loc[indexes, 'mixmatch_index'].tolist()

# save selected mixmatch outputs
mixmatch_path = '/data/hyeryung/mixmatch/output_samples/senti/clsf_pos_merged/opt_samples.jsonl'

mixmatch = pd.read_json(mixmatch_path, lines=True)
# mixmatch.loc[indexes_mixmatch,].to_json('/data/hyeryung/mucoco/outputs/sentiment/mixmatch/clsf_pos_merged/pos.dev_set_below_positive_threshold.jsonl',
                            #    orient='records', lines=True)